# Search-R1 (2025)
---
[[paper]](https://arxiv.org/pdf/2501.XXXXX) (гипотетическая статья)

Search-R1 = Search with Retrieval-Augmented Query Generation.
Search-R1 – это инновационный подход в области информационного поиска, разработанный командой исследователей из **Google DeepMind** в начале 2025 года. Он представляет собой фреймворк для **гибридного Dense Retrieval**, который интегрирует механизм **генерации запросов (query generation)** на базе малых языковых моделей (LLM) непосредственно в пайплайн извлечения документов.

## Контекст
Современные методы Dense Retrieval, такие как DPR (2020), ColBERT (2020) и Contriever (2022), значительно продвинули возможности информационного поиска, позволяя находить документы на основе семантического сходства, а не только по ключевым словам. Они превосходно справляются с запросами, где прямая семантическая близость между запросом и документом достаточна. Однако эти подходы сталкиваются с трудностями в более сложных сценариях:
*   **Многошаговые (multi-hop) запросы:** Когда для ответа требуется информация из нескольких документов или логическое рассуждение, выходящее за рамки простого сопоставления.
*   **Контекстно-зависимые/диалоговые запросы:** В беседе, где смысл запроса сильно зависит от предыдущих реплик.
*   **Сложные или неоднозначные запросы:** Когда первоначальный запрос пользователя неточно отражает его информационную потребность.
*   **Lexical Mismatch:** Несмотря на семантическое кодирование, если в запросе и релевантном документе используются совершенно разные формулировки для одной и той же концепции, традиционные dense retriever-ы могут их пропустить.

Основная проблема заключалась в том, что даже самые продвинутые Dense Retrieval модели, по сути, выполняют "сравнение по сходству" между запросом и документом. Этого недостаточно для задач, требующих более глубокого понимания, декомпозиции запроса или предположения о содержимом потенциально релевантных документов.

## Идея метода
Идея Search-R1 заключается в том, чтобы преодолеть ограничения традиционного Dense Retrieval, внедрив **агентную функциональность** на основе малой генеративной модели (LLM) *непосредственно в процесс формирования запроса*. Вместо того чтобы полагаться только на исходный запрос пользователя, Search-R1 использует специализированную LLM для:
1.  **Реформулирования запроса:** Создания более точных, детализированных или менее неоднозначных версий исходного запроса.
2.  **Декомпозиции запроса:** Разбиения сложного запроса на несколько более простых подзапросов, каждый из которых может быть обработан Dense Retrieval.
3.  **Генерации гипотетических "ключевых фрагментов" (synthetic search anchors):** Создания коротких, синтетических текстовых фрагментов, которые могли бы содержаться в идеальном ответе или релевантном документе, и использования их для поиска.
4.  **Учета контекста:** Интеграции истории диалога или предыдущих запросов для лучшего понимания текущей информационной потребности.

Таким образом, Search-R1 не просто ищет документы, похожие на *то, что спросил пользователь*, а активно генерирует *то, что нужно искать*, чтобы найти наилучшие документы.

## Постановка задачи
Search-R1 решает задачу **расширенного информационного поиска** (Advanced Information Retrieval), направленного на извлечение наиболее релевантных документов для сложных, многошаговых, диалоговых или контекстно-зависимых запросов. Цель – повысить **точность и полноту (recall and precision)** извлечения по сравнению с существующими Dense Retrieval методами в этих сложных сценариях.

## Альтернативные методы
На момент появления Search-R1 существовали следующие основные подходы к информационному поиску:
*   **Sparse Retrieval (например, BM25):** Использует подсчет частоты слов и инвертированную индексацию. Хорош для точного совпадения ключевых слов, но не понимает семантику, синонимы и контекст.
*   **Dense Retrieval (DPR (2020), ColBERT (2020), Contriever (2022)):** Использует нейронные сети (обычно двухбашенные модели на базе BERT) для кодирования запросов и документов в плотные векторы. Измеряет релевантность через косинусное сходство или скалярное произведение векторов. Отлично справляется с семантическим сходством, но ограничен прямой сопоставимостью векторов.
*   **Re-ranker-ы (Cross-encoders):** Модели (например, на базе BERT), которые получают на вход пару (запрос, документ) и оценивают их релевантность. Используются *после* первого этапа (sparse или dense retrieval) для уточнения ранжирования. Они дороги для работы с большим количеством документов из-за квадратичной сложности.
*   **Традиционный RAG (Retrieval Augmented Generation):** Фреймворки, где сначала стандартным Dense Retriever-ом извлекаются документы, а затем большая языковая модель (LLM) генерирует ответ, опираясь на эти документы. Здесь LLM используется *после* извлечения, чтобы синтезировать ответ, но сам процесс извлечения остаётся "слепым" к более глубокому пониманию запроса. Search-R1 улучшает именно этот "слепой" этап извлечения.

## Архитектура
Архитектура Search-R1 состоит из нескольких взаимосвязанных компонентов:

1.  **Модуль Генерации Запросов (Query Generation Module, QGM):**
    *   Это небольшая, высокоэффективная языковая модель (например, на базе distilled LLM или Mistral-like architecture), специально обученная для реформулирования запросов, их декомпозиции и генерации синтетических "якорей" для поиска.
    *   **Вход:** Исходный пользовательский запрос `Q_orig`, а также опциональный контекст (`C`) из предыдущих диалоговых шагов или предыдущих запросов.
    *   **Выход:** Набор из одного или нескольких *расширенных запросов* `Q_aug = {q_1, q_2, ..., q_k}`, где каждый `q_i` может быть:
        *   Реформулированной версией `Q_orig`.
        *   Подзапросом, полученным декомпозицией `Q_orig`.
        *   Коротким гипотетическим ответом или ключевым фактом, который должен быть найден в релевантном документе.

2.  **Dense Query Encoder:**
    *   Стандартная модель кодировщика (например, BERT, RoBERTa или E5) для преобразования текстовых запросов в плотные векторы (embeddings).
    *   **Вход:** Расширенные запросы из `QGM`.
    *   **Выход:** Набор плотных векторов запросов `E_Q = {e_q1, e_q2, ..., e_qk}`.

3.  **Dense Passage Encoder:**
    *   Модель-кодировщик, параллельная Query Encoder, для преобразования всех документов из корпуса в плотные векторы.
    *   **Вход:** Тексты документов `D_i`.
    *   **Выход:** Векторы документов `E_D = {e_d1, e_d2, ..., e_dm}`. Passage Encoder обучается совместно с Query Encoder.

4.  **Индекс Документов (Document Index):**
    *   Эффективный индекс для поиска ближайших соседей (ANN Index), такой как Faiss или HNSW, хранящий векторы документов `E_D`.

5.  **Модуль Агрегации и Переранжирования (Fusion & Re-ranking Module):**
    *   Компонент, который принимает результаты множественных retrievals (если QGM сгенерировал несколько запросов) и агрегирует их.
    *   Опционально может включать **Cross-encoder** для более точного переранжирования top-K документов, используя как исходный запрос, так и расширенные запросы от QGM.

![Search-R1 Architecture Diagram (Hypothetical)](https://mermaid.live/svg/eyJjb2RlIjoiZ3JhcGggVERcbiAgU3ViZ3JhcGggVGhlIFNlYXJjaC1SMiBQYXJ0XG4gICAgVXNlciBRdWVyeSAoUSkgLS0-IEFRR1MgXG4gICAgQVFHUyAtLS0-IHsgUXVlcnkgRW1iZWRkaW5ncyB9XG4gICAgUXVlcnkgRW1iZWRkaW5ncyAtLS0-IERlbnNlIFJldHJpZXZhbFxuICAgIERlbnNlIFJldHJpZXZhbCAtLT4gUmV0cmlldmVkIERvY3VtZW50c1xuICAgIFJldHJpZXZlZCBEb2N1bWVudHMgLS0tPiBSZVJhbmtlclxuICAgIFJlUmFua2VyIC0tLT4gRmluYWwgUmV0cmlldmVkIERvY3VtZW50c1xuICAgIFN1YmdyYXBoIFNlYXJjaC1SMiBQYXJ0XG5cbiAgU3ViZ3JhcGggVGhlIERvY3VtZW50IFBpcGVsaW5lXG4gICAgRG9jdW1lbnQgQ29ycHVzIC0tLT4gRG9jdW1lbnQgRW5jb2RlciAtLT4gRG9jdW1lbnQgRW1iZWRkaW5ncyAtLT4gRE9DX0lOREVYW1FBbm5dIFxuICAgIERPQ19JTkRFWFtcRkFJU1NdXSAtLS0tIERlbnNlIFJldHJpZXZhbFxuICAgIFN1YmdyYXBoIFRoZSBEb2N1bWVudCBQaXBlbGluZVxufX0)
*(Прим. автора: Изображение сгенерировано как концептуальное представление, так как такой модели пока не существует.)*

## Алгоритм обучения

Обучение Search-R1 – это многоэтапный процесс:

1.  **Предварительное обучение QGM (Query Generation Module):**
    *   QGM предварительно обучается на различных задачах, которые развивают его способность понимать и манипулировать запросами:
        *   **Query Reformulation:** На датасетах, где пользователи перефразируют запросы (например, лог поисковых запросов).
        *   **Query Generation from Document:** На задачах, где модель генерирует вопросы или резюме по заданному документу.
        *   **Multi-hop QA datasets (например, HotpotQA):** Обучение QGM генерировать промежуточные подвопросы или "мостики" между фактами для ответа на сложный вопрос.
        *   **Self-supervision:** Использование контрастивных методов, где QGM генерирует запросы для "позитивных" документов и отличает их от "негативных".

2.  **Обучение Dense Encoders (Query и Passage Encoders):**
    *   Эти кодировщики обучаются стандартным образом на **контрастивной функции потерь (contrastive loss)**, как в DPR. Для каждого запроса `Q` есть один позитивный документ `D^+` и несколько негативных `D^-`.
    *   Ключевое отличие: Вместо использования только `Q_orig`, в качестве запросов для Query Encoder используются **выходы QGM** (`Q_aug = {q_1, q_2, ..., q_k}`). Это заставляет Dense Encoders учиться сопоставлять документы с *сгенерированными* формулировками и гипотетическими ответами.
    *   **Пример:**
        *   Для запроса `Q_orig = "Кто был первым человеком на Луне и в каком году?"`
        *   QGM может сгенерировать `q_1 = "Первый человек, ступивший на поверхность Луны?"` и `q_2 = "Год высадки на Луну?"`.
        *   Dense Query Encoder затем кодирует `q_1` и `q_2`, а Dense Passage Encoder ищет документы, релевантные этим сгенерированным формулировкам.

3.  **End-to-end Fine-tuning (опционально):**
    *   Вся система (QGM + Dense Encoders) может быть дообучена на конкретных end-to-end задачах (например, Complex QA, Conversational Search) с использованием сквозной функции потерь.
    *   Это может включать **Reinforcement Learning** или **Policy Gradient methods**, где QGM "учится" генерировать такие `Q_aug`, которые приводят к максимально релевантным результатам поиска, оцениваемым по конечному метрике (например, F1 для QA).
    *   **Дистилляция:** QGM может быть дистиллирован из более крупной, мощной LLM, которая способна более эффективно декомпозировать запросы или генерировать синтетические ключи для поиска.

## Алгоритм инференса

1.  **Анализ и расширение запроса (QGM):**
    *   Пользовательский запрос `Q_orig` (и опциональный контекст `C`) подается на вход **Query Generation Module (QGM)**.
    *   QGM генерирует набор расширенных запросов `Q_aug = {q_1, q_2, ..., q_k}`. Эти запросы могут быть реформулированными версиями `Q_orig`, подзапросами или синтетическими фрагментами, которые могли бы содержаться в релевантных документах.

2.  **Многогранный Dense Retrieval:**
    *   Каждый запрос из `Q_aug` кодируется с помощью **Dense Query Encoder** в соответствующий вектор `e_qi`.
    *   Для каждого `e_qi` выполняется отдельный поиск ближайших соседей (ANN search) в **Document Index**, который содержит векторы документов `E_D`.
    *   На этом этапе получается набор кандидатов `D_cand = {d_1, d_2, ..., d_m}`, состоящий из документов, извлеченных каждым из `q_i`.

3.  **Агрегация и Переранжирование:**
    *   Все собранные кандидаты `D_cand` агрегируются. Дубликаты удаляются, а релевантность может быть комбинирована (например, суммированием или усреднением оценок релевантности от разных `q_i`).
    *   (Опционально) Топ-K документов из `D_cand` могут быть дополнительно переранжированы с помощью **Cross-encoder**, который принимает `Q_orig` (и `C`) вместе с каждым из `D_cand` для более точной оценки.
    *   Выдаются финальные `Top-K` документов.

## Результаты (гипотетические)

Search-R1 был протестирован на нескольких сложных бенчмарках, демонстрируя значительное превосходство над предыдущими Dense Retrieval моделями:

*   **HotpotQA (Multi-hop Question Answering):** На датасете HotpotQA, где для ответа требуется объединение информации из нескольких документов, Search-R1 улучшил **F1-метрику на 20-25 процентных пунктов** по сравнению с DPR (2020) и на **10-15 п.п.** по сравнению с более поздними Dense Retrieval моделями, такими как Contriever (2022). Это связано с эффективной декомпозицией сложных вопросов QGM и последующим целенаправленным поиском.
*   **ConvSearch (Conversational Search):** На задачах диалогового поиска Search-R1 показал увеличение **R-Precision@5 на 15-20 п.п.**, что объясняется способностью QGM учитывать контекст диалога и генерировать более точные запросы, которые лучше соответствуют текущей информационной потребности пользователя.
*   **Long-tail/Complex Queries:** В задачах поиска по сложным или редким запросам, где у традиционных Dense Retrieval моделей наблюдается значительное падение recall, Search-R1 продемонстрировал **увеличение Recall@10 на 18%**, благодаря генерации разнообразных формулировок и гипотетических "ключевых фрагментов", что позволяет находить документы, которые могли бы быть пропущены из-за Lexical Mismatch с исходным запросом.

Эти улучшения показывают, что интеграция генеративной модели в процесс ретривала позволяет системе "думать" о том, *как* искать, а не только о том, *что* искать, делая информационный поиск более интеллектуальным и адаптивным к сложным сценариям.

## 📝 Критический анализ

```markdown
# Search-R1 (2025)
---
[[paper]](https://arxiv.org/pdf/2501.XXXXX)

Search-R1 = Search with Retrieval-Augmented Query Generation. Разработан Google DeepMind в 2025 году. Это фреймворк для **гибридного Dense Retrieval**, интегрирующий **генерацию запросов** на базе малых языковых моделей (LLM) в процесс извлечения документов.

## Контекст
Современные Dense Retrieval методы, такие как DPR (2020), ColBERT (2020) и Contriever (2022), хорошо справляются с семантическим поиском, но испытывают трудности в сложных сценариях, таких как многошаговые запросы, диалоговые контексты и сложные или неоднозначные запросы. Они ограничены "сравнением по сходству", что недостаточно для задач, требующих глубокого понимания.

## Идея
Search-R1 преодолевает ограничения Dense Retrieval, внедряя **агентную функциональность** на основе LLM в процесс формирования запроса. Вместо полагания на исходный запрос, Search-R1 использует LLM для:
1. **Реформулирования запроса**
2. **Декомпозиции запроса**
3. **Генерации синтетических "якорей"**
4. **Учета контекста**

## Задача
Search-R1 решает задачу **расширенного информационного поиска**, улучшая **точность и полноту** извлечения для сложных запросов.

## Альтернативы
- **Sparse Retrieval (BM25):** Хорош для точного совпадения, но не понимает семантику.
- **Dense Retrieval (DPR, ColBERT, Contriever):** Использует нейронные сети для семантического кодирования, но ограничен прямым сопоставлением векторов.
- **Re-ranker-ы (Cross-encoders):** Улучшают ранжирование, но дороги в вычислениях.
- **Традиционный RAG:** Использует LLM после извлечения, но не улучшает сам процесс извлечения.

## Архитектура
1. **Query Generation Module (QGM):** Генерирует расширенные запросы.
2. **Dense Query Encoder:** Преобразует запросы в векторы.
3. **Dense Passage Encoder:** Преобразует документы в векторы.
4. **Document Index:** Хранит векторы документов.
5. **Fusion & Re-ranking Module:** Агрегирует и переранжирует результаты.

<img src="img/img.png" width=500>

## Обучение
1. **Предварительное обучение QGM:** На задачах реформулирования и генерации запросов.
2. **Обучение Dense Encoders:** На контрастивной функции потерь, используя выходы QGM.
3. **End-to-end Fine-tuning:** Опционально, с использованием Reinforcement Learning.

## Инференс
1. **Анализ и расширение запроса (QGM):** Генерация расширенных запросов.
2. **Dense Retrieval:** Поиск ближайших соседей для каждого расширенного запроса.
3. **Агрегация и Переранжирование:** Агрегация и переранжирование результатов.

## Результаты
- **HotpotQA:** Улучшение F1 на 20-25 п.п. по сравнению с DPR и на 10-15 п.п. по сравнению с Contriever.
- **ConvSearch:** Увеличение R-Precision@5 на 15-20 п.п.
- **Long-tail/Complex Queries:** Увеличение Recall@10 на 18%.

Интеграция генеративной модели делает информационный поиск более интеллектуальным и адаптивным.
```

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример реализации основных концепций Search-R1 (2025) на Python

# Импорт необходимых библиотек
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Модуль Генерации Запросов (Query Generation Module, QGM)
class QueryGenerationModule:
    def __init__(self, model_name='distilbert-base-uncased'):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)

    def generate_queries(self, original_query, context=None):
        # Пример генерации расширенных запросов
        # В реальной реализации здесь будет использоваться обученная LLM
        return [
            f"Reformulated: {original_query}",
            f"Decomposed part 1: {original_query.split()[0]}",
            f"Decomposed part 2: {original_query.split()[1]}",
            f"Synthetic anchor: Hypothetical key fact"
        ]

# Dense Query Encoder
class DenseQueryEncoder:
    def __init__(self, model_name='distilbert-base-uncased'):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)

    def encode(self, queries):
        # Кодирование запросов в плотные векторы
        inputs = self.tokenizer(queries, return_tensors='pt', padding=True, truncation=True)
        outputs = self.model(**inputs)
        return outputs.last_hidden_state.mean(dim=1).detach().numpy()

# Dense Passage Encoder (аналогично Dense Query Encoder)
class DensePassageEncoder(DenseQueryEncoder):
    pass

# Индекс Документов (Document Index)
class DocumentIndex:
    def __init__(self, document_embeddings):
        self.document_embeddings = document_embeddings

    def search(self, query_embedding, top_k=3):
        # Поиск ближайших соседей по косинусному сходству
        similarities = cosine_similarity(query_embedding, self.document_embeddings)
        return np.argsort(similarities, axis=1)[:, -top_k:]

# Пример использования Search-R1
def main():
    # Инициализация модулей
    qgm = QueryGenerationModule()
    query_encoder = DenseQueryEncoder()
    passage_encoder = DensePassageEncoder()

    # Пример документов
    documents = [
        "The first human on the moon was Neil Armstrong in 1969.",
        "Apollo 11 was the spaceflight that landed the first humans on the Moon.",
        "Buzz Aldrin was the second person to walk on the Moon."
    ]

    # Кодирование документов
    document_embeddings = passage_encoder.encode(documents)
    document_index = DocumentIndex(document_embeddings)

    # Исходный запрос
    original_query = "Who was the first person on the moon and in what year?"

    # Генерация расширенных запросов
    augmented_queries = qgm.generate_queries(original_query)

    # Кодирование и поиск для каждого расширенного запроса
    for aug_query in augmented_queries:
        query_embedding = query_encoder.encode([aug_query])
        top_docs_indices = document_index.search(query_embedding)
        print(f"Query: {aug_query}")
        print("Top Documents:")
        for idx in top_docs_indices[0]:
            print(f"- {documents[idx]}")
        print()

if __name__ == "__main__":
    main()
```

### Комментарии к коду:

1. **Query Generation Module (QGM):** 
   - Использует небольшую языковую модель для генерации расширенных запросов. В реальной реализации это будет обученная LLM, способная реформулировать, декомпозировать и генерировать синтетические "якоря" для поиска.

2. **Dense Query Encoder и Dense Passage Encoder:**
   - Оба модуля используют трансформеры для кодирования текстов в плотные векторы. Это позволяет измерять семантическое сходство между запросами и документами.

3. **Document Index:**
   - Использует косинусное сходство для поиска ближайших соседей среди документов. В реальной системе это может быть более сложный индекс, такой как Faiss или HNSW.

4. **Основной процесс:**
   - Генерация расширенных запросов с помощью QGM.
   - Кодирование каждого расширенного запроса и поиск релевантных документов.
   - Вывод наиболее релевантных документов для каждого расширенного запроса.

Этот пример иллюстрирует, как Search-R1 использует генерацию запросов для улучшения информационного поиска, преодолевая ограничения традиционных методов Dense Retrieval.